# 🏆 Lesson 16 · Capstone · FieldAdvisor

*“You have learned the pieces. Now build the thing, and be honest about what it can and cannot do.”*

| ⏱ Time | 🎚 Level | 🗓 Suggested day | 📚 You should know |
|:--|:--|:--|:--|
| ~420 min | 🔴 Capstone | Day 14–15 (≈ 2 days) | Everything, especially Projects 3 and 4 |

## 🎯 Learning objectives
By the end of this lesson you will be able to:

- Integrate a **CNN leaf-disease classifier** and an **LSTM/ridge price forecaster** into one decision-support prototype
- Design **uncertainty-aware advice**: triage tiers for diagnoses, prediction intervals for prices, a transparent hold-or-sell rule
- Evaluate the **whole system** with honest, pre-agreed criteria (accuracy, per-class recall, error vs. baseline, interval coverage)
- Write a **model card** documenting data, intended use, metrics, limitations and ethical considerations
- Practise **professional engineering habits**: reproducibility, clean interfaces, tests, communication

## 🗺 Roadmap
1. The vision
2. Brief & deliverables
3. Milestone 1: leaf data
4. Milestone 2: disease model
5. Milestone 3: price forecaster
6. Milestone 4: the advice engine
7. Milestone 5: system evaluation
8. Milestone 6: model card
9. Milestone 7: demo
10. Ethics & responsible deployment
11. Rubric & presentation
12. Stretch tracks

## ⚙️ Setup
Run this cell first. It loads our libraries and the course helper module (`utils/dlcourse.py`).

In [1]:
import sys; sys.path.append("..")          # so we can import the course helper library
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
cpu = torch.device('cpu')
dl.set_seed(42)                          # reproducible randomness
%matplotlib inline

---
## 🌍 1. The vision

> **Wanjiru farms 2 hectares of maize near Nakuru.** On her basic Android phone she opens **FieldAdvisor**.
> She photographs a leaf with odd yellow streaks. The app says: *"This looks like **maize streak virus** (high confidence). Remove and destroy affected plants; consult your extension officer about resistant varieties."*
> It then shows the maize price outlook: *"Nairobi price today: 4,800 KES/bag. In 4 weeks we expect about 5,200 (likely range 4,900–5,600). Suggested action: **HOLD** if you can store safely."*

You will build the **intelligence behind that screen**: two models, one advice engine, honest evaluation and documentation. It is a prototype on **synthetic data**: the learning goal is the *engineering and judgement*, not agronomic truth.

```
leaf photo ──▶ [CNN] ──▶ class probabilities ──▶ ┐
                                                 ├──▶ [Advice engine] ──▶ advice card
price history + rain + fuel ─▶ [forecaster] ─▶ forecast + interval ─▶ ┘
```

---
## 📋 2. Brief & deliverables

| Deliverable | Description | Where |
|:--|:--|:--|
| 1. **Working notebook** | All milestones below complete, passing their self-checks, reproducible (seeds set), runs top-to-bottom | this notebook |
| 2. **Evaluation report** | Test accuracy, per-class recall, price MAE vs naive, interval coverage: plus *error analysis* | Milestone 5 + your written notes |
| 3. **Model card** | One page: intended use, data, metrics, limitations, ethics (template in `../capstone/MODEL_CARD_TEMPLATE.md`) | Milestone 6 |
| 4. **Advice-card demo** | Three example cards produced end-to-end from *test* data | Milestone 7 |
| 5. **5-minute presentation** | Problem → approach → results → limitations → what next (guide in `../capstone/PRESENTATION_GUIDE.md`) | your slides |

**Pre-agreed success criteria (set them *before* you look at results!)**

| Component | Criterion |
|:--|:--|
| Disease classifier | ≥ **85 %** test accuracy; recall ≥ **70 %** on every disease *(the self-check is lenient at 65 %; aim higher)* |
| Price forecaster | 4-week **test MAE ≤ 0.8 × naive**; 80 % interval **coverage between 70 % and 90 %** |
| Advice engine | Passes the behavioural tests; **never gives a confident diagnosis below the confidence threshold** |
| Documentation | Model card covers all required sections with **real numbers** and **honest limitations** |

**Suggested schedule:** Day 1 (M1–M3 (models). Day 2) M4–M7 (system, evaluation, documentation, demo) + presentation.
Each milestone has hints; solutions are in `../solutions/16_capstone_fieldadvisor_solutions.ipynb`: try hard before peeking.

> 🔁 **Run the milestones in order**: later ones use the variables the earlier ones create (`leaf`, `leaf_model`, `fc`, `report`).

In [2]:
# ══════════════ GIVEN: price-forecasting data preparation (you built this in Lesson 12) ══════════════
import pandas as pd
W, H = 52, 4                                                     # 52-week input window → forecast 1..4 weeks ahead
markets = ["Nairobi", "Eldoret", "Kisumu"]
_all = dl.load_prices(None)
series = {}
for _m in markets:
    _df = _all[_all.market == _m].reset_index(drop=True); _doy = _df.date.dt.dayofyear.values
    series[_m] = dict(logp=np.log(_df.price_kes_90kg.values.astype(float)), price=_df.price_kes_90kg.values.astype(float),
                      rain=_df.rainfall_mm.values, fuel=_df.fuel_kes_per_litre.values,
                      sin=np.sin(2 * np.pi * _doy / 365.25), cos=np.cos(2 * np.pi * _doy / 365.25))
dates = pd.to_datetime(_all[_all.market == "Nairobi"].date.values); n = len(dates)
t_tr = int(np.where(dates <= "2020-12-31")[0].max()); t_va = int(np.where(dates <= "2022-12-31")[0].max())
rain_mean = np.mean([series[m]["rain"][:t_tr + 1].mean() for m in markets])      # TRAIN-only statistic

def build(market, split, multi=True):
    """X (N, 52, 5): relative log-price, rain, relative fuel, sin, cos · Y (N, 4): log-changes at +1..+4 weeks · origins (N,)."""
    d = series[market]; lo, hi = {"train": (W - 1, t_tr), "val": (t_tr, t_va), "test": (t_va, n - 1)}[split]
    X, Y, origins = [], [], []
    for t in range(lo, hi - H + 1):
        sl = slice(t - W + 1, t + 1)
        feats = [d["logp"][sl] - d["logp"][t]]
        feats += [d["rain"][sl] / rain_mean, d["fuel"][sl] / d["fuel"][t] - 1, d["sin"][sl], d["cos"][sl]] if multi else [d["sin"][sl], d["cos"][sl]]
        X.append(np.stack(feats, -1)); Y.append(d["logp"][t + 1:t + H + 1] - d["logp"][t]); origins.append(t)
    return np.array(X, np.float32), np.array(Y, np.float32), np.array(origins)

def pooled(split, mkts, multi=True):
    parts = [build(m, split, multi) for m in mkts]
    return np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])

def to_kes(origins, delta, market="Nairobi"):
    return series[market]["price"][origins][:, None] * np.exp(delta)

X_va, Y_va, origins_va = build("Nairobi", "val")
X_te, Y_te, origins_te = build("Nairobi", "test")
print(f"price windows → train/val/test (Nairobi): {len(build('Nairobi', 'train')[1])} / {len(Y_va)} / {len(Y_te)}  | inputs {X_te.shape[1:]} | targets {Y_te.shape[1:]}")

def format_card(card):
    """Pretty-print an advice card (a dict) as a farmer-friendly text box."""
    lines = ["┌" + "─" * 66 + "┐", "│ 🌽 FIELD ADVISOR".ljust(67) + "│", "├" + "─" * 66 + "┤"]
    for key, label in [("diagnosis_text", "🍃 Leaf"), ("price_text", "📈 Price"), ("action", "✅ Action")]:
        txt = str(card.get(key, ""))
        while txt:
            lines.append(f"│ {label if txt == str(card.get(key, '')) else ' ' * len(label)}: {txt[:52]}".ljust(67) + "│"); txt = txt[52:]
    lines.append("└" + "─" * 66 + "┘")
    return "\n".join(lines)

price windows → train/val/test (Nairobi): 467 / 101 / 154  | inputs (52, 5) | targets (4,)


---
## 🎯 3. The milestones

---
### 🧩 Problem 16.1: Milestone 1 - Leaf data pipeline  &nbsp; `⭐`

Write `make_leaf_data(batch_size=64)` that loads the maize-leaf dataset and returns a dict with:

* `train_dl`: a shuffled `DataLoader` of normalised **channels-first** training images and labels,
* `val_dl`: a `DataLoader` for validation (batch 450),
* `X_tr`, `X_te`: the normalised training / test image tensors `(N, 3, 48, 48)`,
* `y_te`: test labels (NumPy), and `class_names`.

Normalise **per colour channel using TRAIN statistics only** (Lesson 10). Then call `leaf = make_leaf_data()`.

<details><summary>💡 Hint 1</summary>

`d = dl.load_leaves()` gives `x_train, y_train, x_val, y_val, x_test, y_test` with images `(N, 48, 48, 3)`.

</details>

<details><summary>💡 Hint 2</summary>

`mean = d['x_train'].astype(np.float64).mean(axis=(0, 1, 2))`, same for `std` (float64 avoids precision loss); normalise then `.transpose(0, 3, 1, 2)` and convert to a float32 tensor.

</details>

<details><summary>💡 Hint 3</summary>

`DataLoader(TensorDataset(X, torch.tensor(y)), batch_size=..., shuffle=True)`.

</details>


In [ ]:
def make_leaf_data(batch_size=64):
    d = dl.load_leaves()
    # TODO: per-channel mean/std from TRAIN only; normalise; channels-first tensors; DataLoaders
    raise NotImplementedError

leaf = make_leaf_data()

In [ ]:
# ▶ self-check (run after you've written your answer)
ok = "leaf" in globals() and isinstance(leaf, dict)
dl.check("returns a dict with the required keys", ok and all(k in leaf for k in ("train_dl", "val_dl", "X_tr", "X_te", "y_te", "class_names")))
dl.check("channels-first test tensor (450, 3, 48, 48)", ok and tuple(leaf["X_te"].shape) == (450, 3, 48, 48))
dl.check("training set is standardised per channel (mean≈0, std≈1)", ok and torch.allclose(leaf["X_tr"].mean((0, 2, 3)), torch.zeros(3), atol=1e-3) and torch.allclose(leaf["X_tr"].std((0, 2, 3)), torch.ones(3), atol=2e-2))
dl.check("test set uses TRAIN statistics (its mean is not exactly 0)", ok and leaf["X_te"].mean((0, 2, 3)).abs().max() > 1e-6)
dl.check("5 class names", ok and len(leaf["class_names"]) == 5)

---
### 🧩 Problem 16.2: Milestone 2 - The disease classifier  &nbsp; `⭐⭐⭐`

Write `train_leaf_model(leaf, epochs=25)` that builds and **trains a CNN** on `leaf["train_dl"]` (validating on `leaf["val_dl"]`) and returns the trained model.
Target: **≥ 85 % test accuracy** and **recall ≥ 65 % on every class** (aim for 70 %+). Use what you learned in Projects 2–3: conv blocks with BatchNorm, dropout, **label-preserving augmentation** (rotations by 90° and flips), Adam with lr ≈ 2e-3.
The starter is a weak linear placeholder: replace it. *(Training takes ~1.5 minutes on a CPU.)*

<details><summary>💡 Hint 1</summary>

Re-use Lesson 10's architecture: three `Conv2d(3×3, padding=1) → BatchNorm2d → ReLU → MaxPool2d(2)` blocks with 16, 32, 64 filters → `Flatten → Dropout(0.3) → Linear(64·6·6, 5)`.

</details>

<details><summary>💡 Hint 2</summary>

Augmentation: wrap the loader in a small class whose `__iter__` applies `torch.rot90(x, k, dims=(1, 2))` and an optional `x.flip(2)` to each image.

</details>

<details><summary>💡 Hint 3</summary>

`dl.fit(model, aug_loader, leaf['val_dl'], nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), 2e-3), epochs=epochs, device=cpu, verbose=False)`. Call `dl.set_seed(0)` before creating the model.

</details>


In [ ]:
def train_leaf_model(leaf, epochs=25):
    # TODO: replace this placeholder with a CNN trained with augmentation (see the hints)
    model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 48 * 48, 5))
    return model

leaf_model = train_leaf_model(leaf)

In [ ]:
# ▶ self-check (run after you've written your answer)
probs = dl.predict_proba(leaf_model, leaf["X_te"], device=cpu).numpy(); preds = probs.argmax(1); y = leaf["y_te"]
acc = (preds == y).mean(); recalls = [(preds[y == c] == c).mean() for c in range(5)]
print("per-class recall:", {n: round(float(r), 2) for n, r in zip(leaf["class_names"], recalls)})
dl.check(f"test accuracy ≥ 85%  (yours: {acc:.1%})", acc >= 0.85, "add augmentation + a 3-block CNN")
dl.check(f"every class recall ≥ 65%  (worst: {min(recalls):.1%})", min(recalls) >= 0.65)
dl.check("model outputs 5 logits", tuple(leaf_model(leaf['X_te'][:2]).shape) == (2, 5))

---
### 🧩 Problem 16.3: Milestone 3 - The price forecaster  &nbsp; `⭐⭐⭐`

Write `forecast()` returning a dict `{"val": ..., "test": ...}` of predicted **log-changes** (shape `(N, 4)`, for +1..+4 weeks) for Nairobi's validation and test windows (`X_va`/`X_te` are given).
Target: **4-week test MAE ≤ 0.8 × the naive forecast's** (naive = zeros).
The starter returns the naive forecast. Options: pick one and justify it in your notes:
**(a)** Ridge regression on the flattened window; **(b)** an LSTM/GRU (Lesson 12), ideally pooled over the three markets and averaged over seeds.

<details><summary>💡 Hint 1</summary>

Ridge: `from sklearn.linear_model import Ridge`; `Xp, Yp = pooled('train', markets)`; `r = Ridge(alpha=1).fit(Xp.reshape(len(Xp), -1), Yp)`; predict on `X.reshape(len(X), -1)`.

</details>

<details><summary>💡 Hint 2</summary>

LSTM: `nn.LSTM(5, 32, batch_first=True)`, dropout, `Linear(32, 4)` on the last step; standardise targets with train std; Huber loss; `dl.EarlyStopper`; `grad_clip=1.0`; `task='regression'` in `dl.fit`.

</details>

<details><summary>💡 Hint 3</summary>

Average several seeds (an ensemble) for a steadier forecast.

</details>


In [ ]:
def forecast():
    # TODO: replace the naive forecast with a trained model (Ridge or LSTM/GRU)
    return {"val": np.zeros_like(Y_va), "test": np.zeros_like(Y_te)}

fc = forecast()

In [ ]:
# ▶ self-check (run after you've written your answer)
ok = isinstance(fc, dict) and fc["test"].shape == Y_te.shape and fc["val"].shape == Y_va.shape
dl.check("returns val and test predictions with the right shapes", ok)
if ok:
    actual = to_kes(origins_te, Y_te)[:, 3]
    mae = lambda d: np.abs(to_kes(origins_te, d)[:, 3] - actual).mean()
    rel = mae(fc["test"]) / mae(np.zeros_like(Y_te))
    print(f"4-week test MAE: {mae(fc['test']):.0f} KES vs naive {mae(np.zeros_like(Y_te)):.0f} KES → relative MAE {rel:.2f}")
    dl.check(f"beats naive: relative MAE ≤ 0.8  (yours: {rel:.2f})", rel <= 0.8, "try Ridge on the flattened windows, or an LSTM with rain+fuel inputs")

---
### 🧩 Problem 16.4: Milestone 4 - The advice engine  &nbsp; `⭐⭐⭐`

Write `advice_card(leaf_probs, class_names, price_now, delta_pred, delta_lo, delta_hi, storage_cost=0.03)` returning a dict. Rules:

**Diagnosis tier** by `p = max(leaf_probs)`: `p ≥ 0.90` → `"diagnose"`; `0.60 ≤ p < 0.90` → `"retake"`; `p < 0.60` → `"refer"`.
* `diagnose`: `card["diagnosis"]` is the predicted class name; `card["diagnosis_text"]` = the matching entry of `DISEASE_ADVICE`.
* `retake`: `card["diagnosis"] = None`; text says to retake the photo and names the **top-2 guesses with percentages**.
* `refer`: `card["diagnosis"] = None`; text asks the farmer to consult the extension officer.

**Price outlook** (4 weeks ahead): `price_forecast = price_now·exp(delta_pred)`, `price_low = price_now·exp(delta_lo)`, `price_high = price_now·exp(delta_hi)`; `card["price_text"]` is a readable sentence containing the rounded forecast and range.

**Action** (`gain = price_forecast/price_now − 1`):
* `gain > storage_cost` **and** `price_low > price_now·(1+storage_cost)` → `"HOLD"`
* `gain > storage_cost` but the lower bound isn't above → `"HOLD (low confidence)"`
* otherwise → `"SELL NOW"`

Include all keys: `tier, diagnosis, diagnosis_text, price_now, price_forecast, price_low, price_high, price_text, action`.

<details><summary>💡 Hint 1</summary>

`names[int(np.argmax(leaf_probs))]` and `float(np.max(leaf_probs))`.

</details>

<details><summary>💡 Hint 2</summary>

Top-2: `idx = np.argsort(-leaf_probs)[:2]`.

</details>

<details><summary>💡 Hint 3</summary>

Check the three action cases in order: HOLD, HOLD (low confidence), SELL NOW.

</details>


In [ ]:
DISEASE_ADVICE = {
    "healthy": "No disease detected. Keep scouting weekly.",
    "gray_leaf_spot": "Looks like GRAY LEAF SPOT. Improve air flow, rotate crops, consider fungicide if spreading; confirm with your extension officer.",
    "northern_leaf_blight": "Looks like NORTHERN LEAF BLIGHT. Use resistant varieties next season; fungicide may help early; confirm with your extension officer.",
    "common_rust": "Looks like COMMON RUST. Usually mild; monitor and spray only if severe; confirm with your extension officer.",
    "maize_streak": "Looks like MAIZE STREAK VIRUS. No cure: remove and destroy affected plants, control leafhoppers, use resistant varieties; confirm with your extension officer.",
}

def advice_card(leaf_probs, class_names, price_now, delta_pred, delta_lo, delta_hi, storage_cost=0.03):
    card = {}
    # TODO: diagnosis tier + text, price outlook, action
    return card

In [ ]:
# ▶ self-check (run after you've written your answer)
names = dl.LEAF_CLASSES
try:
    c1 = advice_card(np.array([.02, .95, .01, .01, .01]), names, 4800, np.log(1.08), np.log(1.05), np.log(1.12))
    dl.check("confident diagnosis → tier 'diagnose' with the class name", c1["tier"] == "diagnose" and c1["diagnosis"] == "gray_leaf_spot")
    dl.check("diagnosis text comes from DISEASE_ADVICE", c1["diagnosis_text"] == DISEASE_ADVICE["gray_leaf_spot"])
    c2 = advice_card(np.array([.70, .25, .02, .02, .01]), names, 4800, 0, 0, 0)
    dl.check("0.6 ≤ p < 0.9 → 'retake', no diagnosis, names the top-2 guesses", c2["tier"] == "retake" and c2["diagnosis"] is None and "healthy" in c2["diagnosis_text"] and "gray_leaf_spot" in c2["diagnosis_text"])
    c3 = advice_card(np.array([.40, .30, .15, .10, .05]), names, 4800, 0, 0, 0)
    dl.check("p < 0.6 → 'refer' and NO diagnosis (never confident when unsure)", c3["tier"] == "refer" and c3["diagnosis"] is None)
    dl.check("price numbers: forecast = price_now·exp(Δ)", np.isclose(c1["price_forecast"], 4800 * 1.08) and np.isclose(c1["price_low"], 4800 * 1.05) and np.isclose(c1["price_high"], 4800 * 1.12))
    dl.check("+8% forecast, lower bound above the cost threshold → HOLD", c1["action"] == "HOLD")
    c4 = advice_card(np.array([.97, .01, .01, .005, .005]), names, 4800, np.log(1.08), np.log(0.97), np.log(1.15))
    dl.check("+8% but wide interval → HOLD (low confidence)", c4["action"] == "HOLD (low confidence)")
    c5 = advice_card(np.array([.97, .01, .01, .005, .005]), names, 4800, np.log(1.01), np.log(0.98), np.log(1.04))
    dl.check("+1% (< storage cost) → SELL NOW", c5["action"] == "SELL NOW")
    dl.check("all required keys are present", all(k in c1 for k in ("tier", "diagnosis", "diagnosis_text", "price_now", "price_forecast", "price_low", "price_high", "price_text", "action")))
except Exception as e:
    dl.check("advice_card runs on the test cases", False, f"{type(e).__name__}: {e}")

---
### 🧩 Problem 16.5: Milestone 5 - Evaluate the whole system  &nbsp; `⭐⭐`

Write `system_report(leaf_model, leaf, fc)` returning a dict with:
`test_acc`, `per_class_recall` (list of 5), `mae_4w`, `naive_mae_4w`, `rel_mae_4w` (all in KES, on the **test** set, 4-week horizon),
`q_lo`, `q_hi` (the 10th and 90th percentiles of **validation** residuals `Y_va − fc["val"]`, per horizon; array of 4) and `coverage_4w` (fraction of test 4-week log-changes falling inside `[pred+q_lo[3], pred+q_hi[3]]`).
Then call `report = system_report(leaf_model, leaf, fc)` and **read the numbers critically**.

<details><summary>💡 Hint 1</summary>

Probabilities: `dl.predict_proba(leaf_model, leaf['X_te'], device=cpu).numpy()`; recall per class `c`: `(preds[y == c] == c).mean()`.

</details>

<details><summary>💡 Hint 2</summary>

KES values: `to_kes(origins_te, delta)[:, 3]` for the 4-week column (index 3).

</details>

<details><summary>💡 Hint 3</summary>

Intervals come from the *validation* residuals (not test!): `np.quantile(Y_va - fc['val'], [0.1, 0.9], axis=0)`.

</details>


In [ ]:
def system_report(leaf_model, leaf, fc):
    pass   # return a dict

report = system_report(leaf_model, leaf, fc)

In [ ]:
# ▶ self-check (run after you've written your answer)
ok = isinstance(report, dict) and all(k in report for k in ("test_acc", "per_class_recall", "mae_4w", "naive_mae_4w", "rel_mae_4w", "q_lo", "q_hi", "coverage_4w"))
dl.check("report has all the required keys", ok)
if ok:
    print(f"test acc {report['test_acc']:.1%} | recalls {np.round(report['per_class_recall'], 2)} | 4w MAE {report['mae_4w']:.0f} vs naive {report['naive_mae_4w']:.0f} (rel {report['rel_mae_4w']:.2f}) | 80% interval coverage {report['coverage_4w']:.0%}")
    dl.check("test accuracy ≥ 85%", report["test_acc"] >= 0.85)
    dl.check("worst-class recall ≥ 65%", min(report["per_class_recall"]) >= 0.65)
    dl.check("price forecast beats 0.8 × naive", report["rel_mae_4w"] <= 0.8)
    dl.check("interval coverage between 65% and 92% (nominal 80%)", 0.65 <= report["coverage_4w"] <= 0.92)
    dl.check("relative MAE is consistent with MAE / naive MAE", np.isclose(report["rel_mae_4w"], report["mae_4w"] / report["naive_mae_4w"]))

---
### 🧩 Problem 16.6: Milestone 6 - The model card  &nbsp; `⭐⭐`

A **model card** (Mitchell et al., 2019) is a short document that travels with a model. Write `model_card`: a Markdown **string**: with these sections (use `## ` headings):
**Intended use**, **Data**, **Metrics**, **Limitations**, **Ethical considerations**.
It must contain your **actual numbers** (use the `report` dict and f-strings), at least **three specific limitations**, and **no leftover "TODO"** text. Aim for 200–400 words. A fuller template is in `../capstone/MODEL_CARD_TEMPLATE.md`.

<details><summary>💡 Hint 1</summary>

Use f-strings: `f"Test accuracy: {report['test_acc']:.1%}"`.

</details>

<details><summary>💡 Hint 2</summary>

Limitations to consider: synthetic data; uniform backgrounds; only 5 conditions; small test set; price model trained on 2011–2020 regimes; interval coverage depends on the future resembling validation.

</details>

<details><summary>💡 Hint 3</summary>

Ethical considerations: wrong advice has real costs; who is excluded (no smartphone, language); data consent; human-in-the-loop; monitoring for drift.

</details>


In [ ]:
model_card = f"""
# FieldAdvisor model card

## Intended use
TODO

## Data
TODO

## Metrics
TODO

## Limitations
TODO

## Ethical considerations
TODO
"""

In [ ]:
# ▶ self-check (run after you've written your answer)
txt = model_card if "model_card" in globals() and isinstance(model_card, str) else ""
low = txt.lower()
dl.check("has the five required sections", all(s in low for s in ("## intended use", "## data", "## metrics", "## limitations", "## ethical considerations")))
dl.check("no leftover TODO", "todo" not in low and len(txt) > 0)
dl.check("substantive length (≥ 800 characters)", len(txt) >= 800)
dl.check("contains real numbers (percentages and KES)", txt.count("%") >= 3 and "kes" in low)
lim = low.split("## limitations")[-1].split("## ethical")[0] if "## limitations" in low else ""
dl.check("lists at least three limitations", sum(1 for ln in lim.splitlines() if ln.strip()[:2] in ("1.", "2.", "3.", "- ", "* ")) >= 3)

---
### 🧩 Problem 16.7: Milestone 7 - End-to-end demo  &nbsp; `⭐⭐`

Write `render_demo(leaf_model, leaf, fc, report, n=3)` returning a list of `n` advice cards (dicts from `advice_card`), each built from **one test leaf image and one test price origin**:
pick `n` random test leaves and `n` random test origins with `rng = np.random.default_rng(1)` (`rng.choice(..., n, replace=False)`), use `price_now = series["Nairobi"]["price"][origin]`,
the 4-week forecast `fc["test"][i, 3]` and its interval `fc["test"][i, 3] + report["q_lo"][3]` / `+ report["q_hi"][3]`.
Then print each card with `format_card`.

<details><summary>💡 Hint 1</summary>

`probs = dl.predict_proba(leaf_model, leaf['X_te'], device=cpu).numpy()`

</details>

<details><summary>💡 Hint 2</summary>

Loop `for li, pi in zip(leaf_idx, price_idx)`.

</details>

<details><summary>💡 Hint 3</summary>

Call `advice_card(probs[li], leaf['class_names'], price_now, d, d + report['q_lo'][3], d + report['q_hi'][3])` with `d = fc['test'][pi, 3]`.

</details>


In [ ]:
def render_demo(leaf_model, leaf, fc, report, n=3):
    pass   # return a list of cards

cards = render_demo(leaf_model, leaf, fc, report)
for c in cards or []:
    print(format_card(c))

In [ ]:
# ▶ self-check (run after you've written your answer)
ok = "cards" in globals() and isinstance(cards, list) and len(cards) == 3
dl.check("three cards produced", ok)
dl.check("each card has a tier, diagnosis text, price text and action", ok and all(all(k in c for k in ("tier", "diagnosis_text", "price_text", "action")) for c in cards))
dl.check("actions are valid", ok and all(c["action"] in ("HOLD", "HOLD (low confidence)", "SELL NOW") for c in cards))
dl.check("no confident diagnosis below the threshold", ok and all(c["tier"] == "diagnose" or c["diagnosis"] is None for c in cards))

---
## ⚖️ 4. Ethics & responsible deployment

A system that advises farmers carries real consequences. Before you present, **discuss each question in writing** (3–5 sentences each). They are part of the rubric.

1. **Who is harmed by a wrong answer?** A false *"healthy"* on maize streak vs. a false *"rust"* on a healthy plant: which costs more, and does your thresholding reflect that?
2. **Who is left out?** Farmers without smartphones, with low literacy, speaking languages the app doesn't, or growing varieties/diseases outside the training data.
3. **Data and consent.** If real farmers' photos and prices train the model: who owns the data? Who benefits? Were they asked?
4. **Over-trust.** People trust confident machines. How does your UI communicate uncertainty and encourage confirmation with an expert?
5. **Distribution shift & monitoring.** New diseases, new phone cameras, a drought outside the training years: how would you *notice* the model degrading?
6. **Market effects.** If many farmers follow the same price advice, can the advice change the market it predicts?
7. **Accountability.** Who is responsible when the advice is wrong: the developer, the cooperative, the farmer? What should the app say about its own limits?

See also `../capstone/ETHICS_CHECKLIST.md`.

---
## 📊 5. Rubric & presentation

| Category | Points | What full marks look like |
|:--|:-:|:--|
| **Disease model** | 20 | Meets criteria; justified architecture/augmentation; per-class recall reported; error analysis (which classes, what severity) |
| **Price model** | 20 | Beats naive; compared against ≥ 2 baselines; chronological split with no leakage; ensembles/regularisation justified |
| **Uncertainty & advice engine** | 15 | Triage tiers, calibrated intervals with measured coverage, transparent hold/sell rule, behavioural tests pass |
| **Evaluation rigour** | 15 | Test used once; honest about noise/small samples; ablation or at least one controlled comparison |
| **Model card & ethics** | 15 | All sections, real numbers, ≥ 3 specific limitations, thoughtful ethics answers |
| **Code quality & reproducibility** | 10 | Seeds set, clean functions, runs top-to-bottom, comments explain *why* |
| **Communication** | 5 | Clear 5-minute talk: problem → approach → results → limits → next steps |
| **Total** | **100** | Details in `../capstone/RUBRIC.md` |

**5-minute talk outline:** (1) the farmer's problem · (2) the two models in one slide each · (3) one results table with baselines · (4) one surprising finding from your error analysis · (5) limitations & what you'd do with real data · (6) the demo card.

---
## 🚀 6. Stretch tracks (choose one or more)

| Track | Challenge |
|:--|:--|
| **A. Real data** | Replace the synthetic leaves with a real maize-disease dataset (e.g. PlantVillage maize folders) using `ImageFolder` and a pretrained ResNet-18 (Lesson 10, Section 10). Compare lab-vs-field performance. |
| **B. Ship it** | Export the disease model to **TensorFlow Lite** (or ONNX), quantise it, and measure size and agreement with the original (Lesson 15). |
| **C. Better uncertainty** | Replace residual-quantile intervals with **quantile-loss** outputs or **conformal prediction**; add **temperature scaling** for the CNN's confidence and plot a reliability diagram. |
| **D. More markets & horizons** | Forecast Eldoret/Kisumu; extend to 8–12 weeks; add walk-forward validation. |
| **E. Robustness audit** | Blur, darken or add noise to test leaves; find where the classifier breaks; fix with augmentation; document the before/after. |
| **F. Small & offline** | Shrink the CNN below 10k parameters (or distil it) while keeping ≥ 85 % accuracy: for a phone with no internet. |
| **G. Transformer variant** | Replace the LSTM with a small Transformer encoder (Lesson 13) on the price windows; does it beat ridge? Be honest about the result. |

---
## 🏁 Key takeaways
- ✅ A real system is **models + interfaces + uncertainty + evaluation + documentation**: the model is perhaps a third of the work.
- ✅ **Abstaining** (retake / refer / low-confidence labels) is a feature; confident wrong answers are the failure to design against.
- ✅ Calibrate uncertainty on **validation**, verify it on **test**, and report **coverage**: not just accuracy.
- ✅ Pre-agree **success criteria**, use the test set **once**, and report limitations with the same energy as results.
- ✅ A **model card** and an **ethics review** are part of the deliverable, not an afterthought.
- ✅ You can now train, evaluate, explain and ship basic deep-learning models in **PyTorch** (and read **Keras/TensorFlow**): congratulations! 🎉

---
## 📚 Go deeper (optional)
- Mitchell et al. (2019): *Model Cards for Model Reporting*
- Google PAIR: *People + AI Guidebook* (designing for uncertainty and trust)
- Chip Huyen: *Designing Machine Learning Systems*
- Fast.ai: *Practical Deep Learning for Coders* (next steps: bigger models, real data)
- Stanford CS231n & CS224n; Hugging Face courses: natural next stops after this course

---
⬅️ Previous: [15 · Keras & TensorFlow Bridge](15_keras_tensorflow_bridge.ipynb)